# Data quality

Run `.\.tools\uv.exe run python -m kalix.prepare` from the project root before opening this notebook.
Inspect the common-range working copies and quality report before choosing model-specific rules.
Every calendar day is retained; no values are interpolated or filtered by quality flags.

Both Nikkaluokta (`nikkaluokta_temperature_c`) and Tarfala (`tarfala_temperature_c`) are included.


In [1]:
from kalix.data import load_processed_data, project_root

ROOT = project_root()
daily = load_processed_data(ROOT)
measurements = [
    "nikkaluokta_temperature_c",
    "tarfala_temperature_c",
    "precipitation_mm",
    "discharge_m3_s",
]

In [2]:
daily.head()

,nikkaluokta_temperature_c,nikkaluokta_temperature_quality,nikkaluokta_temperature_start_utc,nikkaluokta_temperature_end_utc,nikkaluokta_temperature_missing,precipitation_mm,precipitation_quality,precipitation_start_utc,precipitation_end_utc,precipitation_missing,discharge_m3_s,discharge_quality,discharge_missing,tarfala_temperature_c,tarfala_temperature_quality,tarfala_temperature_start_utc,tarfala_temperature_end_utc,tarfala_temperature_missing
date,,,,,,,,,,,,,,,,,,
1995-11-22,-10.1,Y,1995-11-22 00:00:01+00:00,1995-11-23 00:00:00+00:00,False,0.8,G,1995-11-22 06:00:01+00:00,1995-11-23 06:00:00+00:00,False,11.1,G,False,-3.6,Y,1995-11-22 00:00:01+00:00,1995-11-23 00:00:00+00:00,False
1995-11-23,-6.3,Y,1995-11-23 00:00:01+00:00,1995-11-24 00:00:00+00:00,False,1.4,G,1995-11-23 06:00:01+00:00,1995-11-24 06:00:00+00:00,False,11.1,G,False,-1.1,Y,1995-11-23 00:00:01+00:00,1995-11-24 00:00:00+00:00,False
1995-11-24,-2.8,Y,1995-11-24 00:00:01+00:00,1995-11-25 00:00:00+00:00,False,1.0,G,1995-11-24 06:00:01+00:00,1995-11-25 06:00:00+00:00,False,11.1,G,False,-7.3,Y,1995-11-24 00:00:01+00:00,1995-11-25 00:00:00+00:00,False
1995-11-25,-24.5,Y,1995-11-25 00:00:01+00:00,1995-11-26 00:00:00+00:00,False,1.0,G,1995-11-25 06:00:01+00:00,1995-11-26 06:00:00+00:00,False,10.8,G,False,-16.8,Y,1995-11-25 00:00:01+00:00,1995-11-26 00:00:00+00:00,False
1995-11-26,-34.8,Y,1995-11-26 00:00:01+00:00,1995-11-27 00:00:00+00:00,False,0.1,G,1995-11-26 06:00:01+00:00,1995-11-27 06:00:00+00:00,False,10.7,G,False,-18.4,Y,1995-11-26 00:00:01+00:00,1995-11-27 00:00:00+00:00,False


In [3]:
summary = daily[measurements].describe().T
summary["missing_days"] = daily[measurements].isna().sum()
summary["missing_percent"] = daily[measurements].isna().mean() * 100
summary

,count,mean,std,min,25%,50%,75%,max,missing_days,missing_percent
nikkaluokta_temperature_c,11079.0,-1.356061,11.628039,-42.10,-8.70,0.1,8.0,21.9,100,0.894534
tarfala_temperature_c,10539.0,-2.445839,7.777972,-28.00,-8.30,-2.8,3.9,18.5,640,5.725020
precipitation_mm,11149.0,1.436658,3.640724,0.00,0.00,0.1,1.1,50.0,30,0.268360
discharge_m3_s,11179.0,30.739555,34.090920,2.43,6.17,16.3,43.8,260.0,0,0.000000


In [4]:
quality_columns = [name for name in daily.columns if name.endswith("_quality")]
for name in quality_columns:
    print(name)
    print(daily[name].value_counts(dropna=False))
    print()

nikkaluokta_temperature_quality
nikkaluokta_temperature_quality
Y       11079
<NA>      100
Name: count, dtype: int64[pyarrow]

precipitation_quality
precipitation_quality
G       11121
<NA>       30
Y          28
Name: count, dtype: int64[pyarrow]

discharge_quality
discharge_quality
G    10999
O      180
Name: count, dtype: int64[pyarrow]

tarfala_temperature_quality
tarfala_temperature_quality
Y       10539
<NA>      640
Name: count, dtype: int64[pyarrow]



In [5]:
missing_columns = [name for name in daily.columns if name.endswith("_missing")]
daily.loc[daily[missing_columns].any(axis=1), measurements + missing_columns]

,nikkaluokta_temperature_c,tarfala_temperature_c,precipitation_mm,discharge_m3_s,nikkaluokta_temperature_missing,precipitation_missing,discharge_missing,tarfala_temperature_missing
date,,,,,,,,
1998-05-16,10.2,NaN,0.0,23.90,False,False,False,True
1998-05-17,6.5,NaN,0.8,33.50,False,False,False,True
1998-05-18,5.9,NaN,0.0,51.10,False,False,False,True
1998-05-19,3.4,NaN,0.0,82.10,False,False,False,True
1998-05-20,-0.1,NaN,0.1,111.00,False,False,False,True
...,...,...,...,...,...,...,...,...
2024-05-03,NaN,3.0,0.0,5.12,True,False,False,False
2024-05-04,NaN,-1.9,0.0,5.22,True,False,False,False
2024-05-05,NaN,-4.7,0.5,5.35,True,False,False,False


In [ ]:
interval_columns = [name for name in daily.columns if name.endswith("_utc")]
daily[interval_columns].head()

## Before modeling

Review gaps by season and source quality definitions. Daily labels describe different measurement windows; inspect the UTC intervals and discharge date convention before defining forecast inputs.

`daily.attrs["sources"]` contains source metadata. All temperature values currently have flag `Y`; a blanket `G` filter would discard the series. Recent discharge flagged `O` remains separate from missing values.

Keep missing calendar rows and real zero precipitation. Later, require complete input and target windows for each training sample instead of discarding whole years automatically. Incomplete precipitation totals must remain unavailable. Missing winter weather can affect calculated snow storage beyond the gap itself.

No filling is performed here. A later experiment could test bounded temperature gaps of at most three days in separate estimated-value columns; respect forecast-time availability and training/validation boundaries. June 1999 precipitation remains missing.


In [ ]:
from IPython.display import Markdown, display

report_path = ROOT / "data" / "processed" / "quality_report.md"
display(Markdown(report_path.read_text(encoding="utf-8")))